In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import  roc_auc_score

In [2]:
X_train = pd.read_pickle('../datos/entrenamiento/X_train.pkl')
X_test = pd.read_pickle('../datos/entrenamiento/X_test.pkl')
y_train = pd.read_pickle('../datos/entrenamiento/y_train.pkl')
y_test = pd.read_pickle('../datos/entrenamiento/y_test.pkl')

In [3]:
algoritmo = RandomForestClassifier()
grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_leaf': [1, 2, 4]
}

In [4]:
from sklearn.model_selection import GridSearchCV

grid_search = GridSearchCV(algoritmo, 
                           grid, 
                           cv = 5, 
                           scoring = 'roc_auc',
                           n_jobs = -1,
                           verbose = 3)

In [5]:
mejor_modelo = grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 27 candidates, totalling 135 fits


In [6]:
print('Mejor combinación:' , grid_search.best_params_)

pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 1000)
pd.DataFrame(grid_search.cv_results_)[['params', 'mean_test_score']].sort_values('mean_test_score', ascending = False)

Mejor combinación: {'max_depth': 20, 'min_samples_leaf': 2, 'n_estimators': 200}


,params,mean_test_score
13,"{'max_depth': 20, 'min_samples_leaf': 2, 'n_estimators': 200}",0.922783
17,"{'max_depth': 20, 'min_samples_leaf': 4, 'n_estimators': 300}",0.922767
16,"{'max_depth': 20, 'min_samples_leaf': 4, 'n_estimators': 200}",0.922715
26,"{'max_depth': None, 'min_samples_leaf': 4, 'n_estimators': 300}",0.922549
14,"{'max_depth': 20, 'min_samples_leaf': 2, 'n_estimators': 300}",0.922524
15,"{'max_depth': 20, 'min_samples_leaf': 4, 'n_estimators': 100}",0.922376
22,"{'max_depth': None, 'min_samples_leaf': 2, 'n_estimators': 200}",0.922050
25,"{'max_depth': None, 'min_samples_leaf': 4, 'n_estimators': 200}",0.922034
23,"{'max_depth': None, 'min_samples_leaf': 2, 'n_estimators': 300}",0.921900
2,"{'max_depth': 10, 'min_samples_leaf': 1, 'n_estimators': 300}",0.921747


In [7]:
rf = RandomForestClassifier(max_depth=None, min_samples_leaf=4, n_estimators=100, random_state=42)

rf.fit(X_train, y_train)

pred= rf.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, pred)
print("ROC AUC Score:", auc)

ROC AUC Score: 0.9196240426547228


In [8]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

metricas_cv = cross_val_score(estimator=rf, 
                              X=X_train, 
                              y=y_train, 
                              cv=skf, 
                              scoring='roc_auc')
print("ROC AUC Score (Cross-Validation):", metricas_cv.mean())

ROC AUC Score (Cross-Validation): 0.9206301929355567
